In [5]:

import os
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import mplcursors
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
import matplotlib
matplotlib.use("Qt5Agg")


In [9]:
def get_correlation(df_processed, result_path, method):
    for idx, row in tqdm(df_processed.iterrows(), total = len(df_processed)):
        file_path = row["FilePath"]
        focus= row["Focus"]
        data = row["Data"]
        freq_band= row["FrequencyBand"]
        df_brain_waves = pd.read_csv(file_path).drop("Unnamed: 0", axis=1, errors="ignore")
        print(f'Reading {file_path}')

        if focus == 'Algorithm': 
            x = df_brain_waves.drop(['Participant', 'SkillScore', 'SkillLevel', 'Algorithm'] , axis=1)
        else:
            x = df_brain_waves.drop(['Participant', 'SkillScore', 'SkillLevel', 'Channel'] , axis=1)

        # Encode 'SkillLevel' using LabelEncoder (Expert as 0, Intermediate as 1 and Novice as 2)
        le = LabelEncoder()
        df_brain_waves['SkillLevelEncoded'] = le.fit_transform(df_brain_waves['SkillLevel'])

        #skill_mapping= {0: 'Expert', 1: 'Intermediate', 2: 'Novice'}

        df_corr= pd.DataFrame(columns=['Feature', 'Correlation', 'P-value'])

        #Correlation Calcuation
        for feature in x.columns:
            # Calculate Spearman correlation
            corr_value, p_value = spearmanr( df_brain_waves['SkillLevelEncoded'], x[feature])
        
            #Save the result into a dataframe
            df_corr.loc[len(df_corr)]= [feature, corr_value, p_value]
        

            print(f"{feature} - Correlation: {corr_value:.4f}, p-value: {p_value:.4f}")

        df_corr.to_csv(result_path+"/"+focus+"_"+data+"_"+freq_band+".csv")

        #Sort the DataFrame by correlation values
        df_corr = df_corr.sort_values(by='Correlation', ascending=False)
        image_path = result_path + "/"+focus+"_"+data+"_"+freq_band+".png"

        # Plotting
        plt.figure(figsize=(14, 8))
        ax = sns.barplot(x='Correlation', y='Feature', data=df_corr, palette='viridis')

        # Display p-value of each feature on plot
        for i, (corr, p_value) in enumerate(zip(df_corr['Correlation'], df_corr['P-value'])):
            ax.text(corr + 0.01, i, f'{p_value:.3f}', va='center', ha='left', color='black')

        #Highlight the significant features
        for i, p_value in enumerate(df_corr['P-value']):
            if p_value < 0.05:
                ax.get_yticklabels()[i].set_fontweight('bold')

        plt.title(f"Spearman Correlation with Skill Level ( {focus} specific analysis on {freq_band} of {data} data using {method})")
        plt.xlabel('Spearman Correlation')
        plt.ylabel('Feature')
        plt.savefig(image_path)
        plt.show()
        print(f"Barplot with correlation values saved to: {image_path}")



### Data Processed With Baseline Correction Algorithm

In [10]:
method = 'Baseline Correction Algorithm'
# create folder to store results if not exist
result_path_bca= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/BCA"
if not os.path.exists(result_path_bca):
    os.makedirs(result_path_bca)

df_processed_bca = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/BCA/processed_data.csv").drop("Unnamed: 0", axis=1, errors="ignore")
get_correlation(df_processed_bca, result_path_bca, method)

  0%|          | 0/4 [00:00<?, ?it/s]

Reading C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/BCA\features_alg_4to50hzbw_ccdata.csv
Mean - Correlation: 0.0447, p-value: 0.1434
StdDev - Correlation: -0.0529, p-value: 0.0835
Skewness - Correlation: 0.1005, p-value: 0.0010
Kurtosis - Correlation: -0.0910, p-value: 0.0029
Variance - Correlation: -0.0529, p-value: 0.0835
ZeroCrossRate - Correlation: -0.0639, p-value: 0.0364
Entropy - Correlation: -0.6111, p-value: 0.0000
Median - Correlation: 0.0329, p-value: 0.2814


C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_31000\3473326573.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x='Correlation', y='Feature', data=df_corr, palette='viridis')


KeyboardInterrupt: 

### Data Processed With Common Average Referencing

In [4]:
method = 'Common Average Referencing'
# create folder to store results if not exist
result_path_car= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/CAR"
if not os.path.exists(result_path_car):
    os.makedirs(result_path_car) 

df_processed_ar = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing/processed_data.csv").drop("Unnamed: 0", axis=1, errors="ignore")
get_correlation(df_processed_ar, result_path_car, method)

  0%|          | 0/8 [00:00<?, ?it/s]

Reading C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing\features_alg_4to50hzbw_bldata.csv
Mean - Correlation: 0.1632, p-value: 0.0000
StdDev - Correlation: 0.0880, p-value: 0.0039
Skewness - Correlation: -0.0105, p-value: 0.7325
Kurtosis - Correlation: 0.0801, p-value: 0.0087
Variance - Correlation: 0.0880, p-value: 0.0039
ZeroCrossRate - Correlation: 0.0199, p-value: 0.5158
Entropy - Correlation: -0.6104, p-value: 0.0000
Median - Correlation: 0.1596, p-value: 0.0000


C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43064\3473326573.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x='Correlation', y='Feature', data=df_corr, palette='viridis')


Barplot with correlation values saved to: C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/CAR/Algorithm_Baseline_4to50hz.png
Reading C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing\features_alg_4to50hzbw_ccdata.csv
Mean - Correlation: 0.1513, p-value: 0.0000
StdDev - Correlation: 0.0047, p-value: 0.8788
Skewness - Correlation: 0.1485, p-value: 0.0000
Kurtosis - Correlation: -0.1735, p-value: 0.0000
Variance - Correlation: 0.0047, p-value: 0.8788
ZeroCrossRate - Correlation: -0.0157, p-value: 0.6068
Entropy - Correlation: -0.5911, p-value: 0.0000
Median - Correlation: 0.1441, p-value: 0.0000


C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43064\3473326573.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x='Correlation', y='Feature', data=df_corr, palette='viridis')


...

Barplot with correlation values saved to: C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/CAR/ElectrodePosition_CodeComprehension_4to50hz.png
Reading C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing\features_elec_abgtbw_bldata.csv
Alpha_Mean - Correlation: 0.2491, p-value: 0.0000
Beta_Mean - Correlation: -0.0568, p-value: 0.0057
Gamma_Mean - Correlation: -0.1298, p-value: 0.0000
Theta_Mean - Correlation: -0.0275, p-value: 0.1814
Alpha_StdDev - Correlation: 0.2344, p-value: 0.0000
Beta_StdDev - Correlation: -0.0470, p-value: 0.0223
Gamma_StdDev - Correlation: -0.1624, p-value: 0.0000
Theta_StdDev - Correlation: -0.0232, p-value: 0.2583
Alpha_Skewness - Correlation: -0.0541, p-value: 0.0085
Beta_Skewness - Correlation: -0.0030, p-value: 0.8835
Gamma_Skewness - Correlation: -0.0728, p-value: 0.0004
Theta_Skewness - Correlation: -0.0054, p-value: 0.7911
Alpha_Kurtosis - Correlation: -0.0371, p-value: 0.0713
Beta_Kurtosis - Correlation: 0.0162, p-value:

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43064\3473326573.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x='Correlation', y='Feature', data=df_corr, palette='viridis')


Barplot with correlation values saved to: C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/CAR/ElectrodePosition_Baseline_AlphaBetaThetaGamma.png
Reading C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Processing\features_elec_abgtbw_ccdata.csv
Alpha_Mean - Correlation: 0.2485, p-value: 0.0000
Beta_Mean - Correlation: -0.0341, p-value: 0.0971
Gamma_Mean - Correlation: -0.1277, p-value: 0.0000
Theta_Mean - Correlation: 0.0325, p-value: 0.1142
Alpha_StdDev - Correlation: 0.2384, p-value: 0.0000
Beta_StdDev - Correlation: -0.0380, p-value: 0.0647
Gamma_StdDev - Correlation: -0.1878, p-value: 0.0000
Theta_StdDev - Correlation: -0.0131, p-value: 0.5233
Alpha_Skewness - Correlation: 0.0469, p-value: 0.0224
Beta_Skewness - Correlation: 0.1016, p-value: 0.0000
Gamma_Skewness - Correlation: -0.0518, p-value: 0.0116
Theta_Skewness - Correlation: -0.0290, p-value: 0.1587
Alpha_Kurtosis - Correlation: -0.0650, p-value: 0.0015
Beta_Kurtosis - Correlation: -0.0122, p-value

C:\Users\Mahima Acharya\AppData\Local\Temp\ipykernel_43064\3473326573.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x='Correlation', y='Feature', data=df_corr, palette='viridis')


Barplot with correlation values saved to: C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/CAR/ElectrodePosition_CodeComprehension_AlphaBetaThetaGamma.png


### Compare

In [4]:
result_path_bca= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/BCA"
result_path_car= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation/CAR"

In [5]:
# Load Correlation DataFrames from BCA folders
df_corr_bca_1 = pd.read_csv(result_path_bca+'/ElectrodePosition_CodeComprehension_AlphaBetaThetaGamma.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_corr_bca_2 = pd.read_csv(result_path_bca+'/Algorithm_CodeComprehension_AlphaBetaThetaGamma.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_corr_bca_3 = pd.read_csv(result_path_bca+'/ElectrodePosition_CodeComprehension_4to50hz.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_corr_bca_4 = pd.read_csv(result_path_bca+'/Algorithm_CodeComprehension_4to50hz.csv').drop("Unnamed: 0", axis=1, errors="ignore")

# Load Correlation DataFrames from AR folders
df_corr_ar_1 = pd.read_csv(result_path_car+'/ElectrodePosition_CodeComprehension_AlphaBetaThetaGamma.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_corr_ar_2 = pd.read_csv(result_path_car+'/Algorithm_CodeComprehension_AlphaBetaThetaGamma.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_corr_ar_3 = pd.read_csv(result_path_car+'/ElectrodePosition_CodeComprehension_4to50hz.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_corr_ar_4 = pd.read_csv(result_path_car+'/Algorithm_CodeComprehension_4to50hz.csv').drop("Unnamed: 0", axis=1, errors="ignore")
common_column = 'Feature'

# Merge DataFrames on the common column
df_merged_1 = pd.merge(df_corr_bca_1, df_corr_ar_1, on=common_column, suffixes=('_bca_ep_seperate', '_ar_ep_seperate'))
df_merged_2 =  pd.merge( df_corr_bca_2,df_corr_ar_2, on=common_column, suffixes=('_bca_alg_seperate','_ar_alg_seperate'))
df_merged_3 =  pd.merge( df_corr_bca_3,df_corr_ar_3, on=common_column, suffixes=('_bca_ep_full','_ar_ep_full'))
df_merged_4 =  pd.merge( df_corr_bca_4,df_corr_ar_4, on=common_column, suffixes=('_bca_alg_full','_ar_alg_full'))

In [6]:
result_path= "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/RQ1/Correlation"
if not os.path.exists(result_path):
    os.makedirs(result_path) 

In [20]:

# Plotting side-by-side bar chart
plt.figure(figsize=(10, 6))

# Assuming the correlation values are stored in columns named 'Correlation_folder1' and 'Correlation_folder2'



plt.bar(df_merged_1['Feature'], df_merged_1['Correlation_bca_ep_seperate'],  label='Electrode Specific Analysis on Alpha, Beta, Theta and Gamma waves using Baseline Correction Algorithm', width=0.35)
plt.bar(df_merged_1['Feature'], df_merged_1['Correlation_ar_ep_seperate'], label='Electrode Specific Analysis on Alpha, Beta, Theta and Gamma waves using Common Average Referencing', width=0.35)
plt.bar(df_merged_2['Feature'], df_merged_2['Correlation_bca_alg_seperate'],label='Algorithm Specific Analysis on Alpha, Beta, Theta and Gamma waves using Baseline Correction Algorithm',width=0.35)
plt.bar(df_merged_2['Feature'], df_merged_2['Correlation_ar_alg_seperate'], label='Algorithm Specific Analysis on Alpha, Beta, Theta and Gamma waves using Common Average Referencing', width=0.35) 
plt.bar(df_merged_3['Feature'], df_merged_3['Correlation_bca_ep_full'], label='Electrode Specific Analysis on 4 to 50Hz waves using Baseline Correction Algorithm', width=0.35)
plt.bar(df_merged_3['Feature'], df_merged_3['Correlation_ar_ep_full'],  label='Electrode Specific Analysis on 4 to 50Hz waves using Common Average Referencing', width=0.35)
plt.bar(df_merged_4['Feature'], df_merged_4['Correlation_bca_alg_full'],  label='Algorithm Specific Analysis on 4 to 50Hz waves using Baseline Correction Algorihm', width=0.35)
plt.bar(df_merged_4['Feature'], df_merged_4['Correlation_ar_alg_full'], label='Algorithm Specific Analysis on 4 to 50Hz waves using Common Average Referencing', width=0.35) 

plt.xlabel('Feature')
plt.ylabel('Correlation Value')
plt.title('Comparison of Correlation Values ')
plt.legend(loc = 'lower right', bbox_to_anchor= (1.2, -0.5), borderaxespad=0., fontsize='small' , ncol=2)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(result_path + '/comparison(barplot).png')
plt.show()

In [32]:
# side-by-side bar chart
plt.figure(figsize=(10, 6))

sns.lineplot(x='Feature', y='Correlation_bca_ep_seperate', data=df_merged_1, label='Electrode Specific Analysis on Alpha, Beta, Theta and Gamma waves using Baseline Correction Algorithm', marker='o')
sns.lineplot(x='Feature', y='Correlation_ar_ep_seperate', data=df_merged_1, label='Electrode Specific Analysis on Alpha, Beta, Theta and Gamma waves using Common Average Referencing', marker='o')
sns.lineplot(x='Feature', y='Correlation_bca_alg_seperate', data=df_merged_2, label='Algorithm Specific Analysis on Alpha, Beta, Theta and Gamma waves using Baseline Correction Algorithm', marker='o')
sns.lineplot(x='Feature', y='Correlation_ar_alg_seperate', data=df_merged_2, label='Algorithm Specific Analysis on Alpha, Beta, Theta and Gamma waves using Common Average Referencing', marker='o') 
sns.lineplot(x='Feature', y='Correlation_bca_ep_full', data=df_merged_3, label='Electrode Specific Analysis on 4 to 50Hz waves using Baseline Correction Algorithm', marker='o')
sns.lineplot(x='Feature', y='Correlation_ar_ep_full', data=df_merged_3, label='Electrode Specific Analysis on 4 to 50Hz waves using Common Average Referencing', marker='o')
sns.lineplot(x='Feature', y='Correlation_bca_alg_full', data=df_merged_4, label='Algorithm Specific Analysis on 4 to 50Hz waves using Baseline Correction Algorithm', marker='o')
sns.lineplot(x='Feature', y='Correlation_ar_alg_full', data=df_merged_4, label='Algorithm Specific Analysis on 4 to 50Hz waves using Common Average Referencing', marker='o') 

plt.xlabel('Features')
plt.ylabel('Correlation')
plt.title('Feature vs. Correlation Comparison')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.legend(loc = 'lower right', bbox_to_anchor= (0.8, -0.4), borderaxespad=0., fontsize='small' , ncol=2)
plt.savefig(result_path + '/comparison(lineplot).png')
plt.show()


: 

# Individual Correlation

In [7]:
df_tmp= pd.read_csv(df_processed_bca["FilePath"][0])
df_tmp

,Unnamed: 0,Participant,SkillScore,SkillLevel,Algorithm,Mean,StdDev,Skewness,Kurtosis,Variance,ZeroCrossRate,Entropy,Median
0,0,1,0.331385,Intermediate,IsPrime,0.623166,0.075672,-0.686589,0.378862,0.005726,0.354839,3.458087,0.640818
1,1,1,0.331385,Intermediate,SiebDesEratosthenes,0.655634,0.073595,-0.364613,-0.554239,0.005416,0.322581,3.459315,0.666387
2,2,1,0.331385,Intermediate,IsAnagram,0.635358,0.074777,-0.381127,-0.654119,0.005592,0.354839,3.458666,0.652463
3,3,1,0.331385,Intermediate,RemoveDoubleChar,0.535681,0.068256,-0.891066,1.496665,0.004659,0.290323,3.457190,0.541280
4,4,1,0.331385,Intermediate,BinToDecimal,0.492292,0.063853,-0.575343,0.486091,0.004077,0.290323,3.457022,0.499086
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,1067,71,0.435651,Expert,DumpSorting,0.656499,0.103989,-0.635405,0.882013,0.010814,0.354839,3.452519,0.687090
1068,1068,71,0.435651,Expert,BinomialCoefficient,0.686300,0.102730,-0.795942,1.710732,0.010553,0.354839,3.453835,0.704360
1069,1069,71,0.435651,Expert,IsAnagram,0.684515,0.101655,-0.732439,1.596281,0.010334,0.354839,3.454073,0.695975
1070,1070,71,0.435651,Expert,ArrayAverage,0.657239,0.093071,-0.224972,1.098488,0.008662,0.354839,3.455448,0.658380


In [23]:
x = df_tmp.drop(['Participant', 'SkillScore', 'SkillLevel', 'Algorithm'] , axis=1)

# Encode 'SkillLevel' using LabelEncoder (Expert as 0, Intermediate as 1 and Novice as 2)
le = LabelEncoder()
df_tmp['SkillLevelEncoded'] = le.fit_transform(df_tmp['SkillLevel'])



In [ ]:

for level, data in groups:
    print(f"Class {level}:")
    for feature in df_tmp.columns:
        if feature != 'SkillLevelEncoded':
            corr, pvalue = spearmanr(data[feature], data['SkillLevelEncoded'])
            print(f"Feature: {feature}, Correaltion: {corr}, p_value: {pvalue}")